# ZestXML benchmarks

Runs every method on **GZ-NPM** and **GZ-Reuters-90** and prints two comparison tables.
Each stage is independent — run stage 0, then whichever of 1–5 you want. The tables in
stage 6 include whatever has been run, so a partial run still gives a partial table.

| stage | cost on an A100 |
|---|---|
| 0 setup + datasets | ~3 min |
| 1 ZestXML and its variants | ~5 min |
| 2 classical baselines | ~5 min |
| 3 SPLADE | ~10 min |
| 4 pretrained encoders | ~10 min |
| 5 Renee | 45–90 min |

Everything but stage 5 also runs on CPU, more slowly. Stage 5 needs a GPU.

## 0 — setup

In [ ]:
!nvidia-smi -L || echo 'no GPU: stages 0-4 still work, stage 5 does not'

In [ ]:
%cd /content
!rm -rf zestxml
!git clone -q -b claude/pytorch-rewrite-fhuwl4 https://github.com/hanialshater/zestxml.git
%cd /content/zestxml
!pip install -q nltk scikit-learn pytest
!python -c "import nltk; nltk.download('reuters', quiet=True)"

# both datasets rebuild from committed snapshots, so they match the CPU runs exactly
!python benchmarks/datasets/make_npm.py GZXML-Datasets/GZ-NPM
!python benchmarks/datasets/make_reuters.py GZXML-Datasets/GZ-Reuters-90
!python -m pytest tests -q | tail -2

## 1 — ZestXML and its variants

The reference configuration per dataset, then the two things that change how an *unseen*
label is reached: a fuzzy direct map, and label feature-bag expansion. Both need word
vectors; GloVe-100d is fetched below.

In [ ]:
# GloVe-100d (gensim-data release asset, ~130 MB). Any word2vec/GloVe/fastText text
# file works -- pass its path as VECTORS.
VECTORS = '/content/glove100.gz'
!wget -q --show-progress -O {VECTORS} \
  https://github.com/RaRe-Technologies/gensim-data/releases/download/glove-wiki-gigaword-100/glove-wiki-gigaword-100.gz
import os; print('vectors:', os.path.exists(VECTORS) and os.path.getsize(VECTORS))

In [ ]:
import os, sys
sys.path.insert(0, '/content/zestxml')
from zestxml import ZestXML

# the reference configuration for each dataset (Results/*2exact/params.txt)
CFG = {
    'GZ-NPM':        dict(shortyK=100, bs_count=40, bs_alpha=0.02, bs_direct_wt=0.8),
    'GZ-Reuters-90': dict(shortyK=50,  bs_count=20, bs_alpha=0.02, bs_direct_wt=0.8),
}
COMMON = dict(bilinear_classifier_cost=5, bilinear_normalize=0, device='auto', num_thread=0)

def run(dataset, tag, **extra):
    m = ZestXML(f'GZXML-Datasets/{dataset}', f'Results/{tag}', **CFG[dataset], **COMMON, **extra)
    m.fit(); m.predict()
    print(f'--- {tag}'); m.evaluate(); print()

for ds, short in (('GZ-NPM', 'npm'), ('GZ-Reuters-90', 'reu')):
    run(ds, f'{short}-reference')
    if os.path.exists(VECTORS):
        run(ds, f'{short}-directmap', direct_map='vectors', direct_vectors=VECTORS,
            direct_topk=3, direct_min_sim=0.5, direct_fallback=1)

In [ ]:
# label feature-bag expansion rebuilds the dataset, so it goes through build_dataset
if os.path.exists(VECTORS):
    for ds in ('GZ-Reuters-90', 'GZ-NPM'):
        cfg = ' '.join(f'{k}={v}' for k, v in CFG[ds].items())
        !python benchmarks/expansion_check.py GZXML-Datasets/{ds} {VECTORS} {cfg} 2>&1 | grep -E 'expansion   :|delta'

## 2 — classical baselines

BM25, kNN and tf-idf centroid (one script, three outputs), and one-vs-all linear.
The point of these is the unseen column: everything except BM25 scores exactly zero there.

In [ ]:
!python benchmarks/baselines/classical.py GZXML-Datasets/GZ-NPM classical 2>&1 | tail -2
!python benchmarks/baselines/classical.py GZXML-Datasets/GZ-Reuters-90 classicalR 2>&1 | tail -2
!python benchmarks/baselines/ova_linear.py -data GZXML-Datasets/GZ-NPM -res_dir Results/ova_linear -epochs 15 2>&1 | tail -2
!python benchmarks/baselines/ova_linear.py -data GZXML-Datasets/GZ-Reuters-90 -res_dir Results/ova_linear_reuters -epochs 15 2>&1 | tail -2

## 3 — SPLADE-style learned sparse

Four ablations. `drop_label_id` removes the per-label feature, which is what makes the
difference between a method that can reach an unseen label and one that cannot.

In [ ]:
for tag, flags in [('splade-base',  '-drop_label_id 0 -norm_labels 0'),
                   ('splade-nodid', '-drop_label_id 1 -norm_labels 0'),
                   ('splade-norm',  '-drop_label_id 0 -norm_labels 1'),
                   ('splade-both',  '-drop_label_id 1 -norm_labels 1')]:
    !python benchmarks/baselines/splade.py -data GZXML-Datasets/GZ-NPM -res Results/{tag} -epochs 6 {flags} 2>&1 | tail -2

## 4 — pretrained encoders

Two questions. Can a sentence encoder retrieve candidates the lexical shortlist misses
(the ceiling on npm is 71.8% recall, and only 54% of *unseen* positives get in)? And is
ZestXML better on a hybrid shortlist than on its own?

In [ ]:
!pip install -q sentence-transformers
!python benchmarks/hf_dense_probe.py --data GZXML-Datasets/GZ-NPM \
    --lexical Results/npm-reference/shortlist.bin --out Results/npm-hybrid \
    --model sentence-transformers/all-MiniLM-L6-v2 --k 100

In [ ]:
# rescore with the hybrid candidates; stage 1 of the model is unchanged, so reuse it
!cp -r Results/npm-reference/model Results/npm-hybrid/model 2>/dev/null || true
m = ZestXML('GZXML-Datasets/GZ-NPM', 'Results/npm-hybrid', **CFG['GZ-NPM'], **COMMON,
            shortlist_file='Results/npm-hybrid/hybrid_shortlist.bin')
m.predict(); m.evaluate()

## 5 — Renee (Microsoft, MLSys 2023)

End-to-end one-vs-all over a transformer encoder, no shortlisting. Needs a GPU and
45–90 min. Three things had to be patched to make it run on a current Colab: `apex` is
not on PyPI (`pip install apex` fetches an unrelated Pyramid library), `batch_encode_plus`
was removed from `transformers`, and the tokenised row counts have to be checked because
Renee maps line N of `trn_X.txt` to row N of `trn_X_Y.txt` without ever verifying it.

In [ ]:
%%bash
set -e
cd /content
[ -d renee/.git ] || git clone -q https://github.com/microsoft/renee.git
pip install -q transformers cython seaborn
pip install -q git+https://github.com/kunaldahiya/pyxclib.git
pip uninstall -y -q apex 2>/dev/null; true

cd /content/renee
cp /content/zestxml/benchmarks/colab/apex.py apex.py     # torch stand-in for the fused optimizers
sed -i 's/tokenizer\.batch_encode_plus(/tokenizer(/' utils/CreateTokenizedFiles.py
mkdir -p xc/Datasets && rm -rf xc/Datasets/GZ-NPM xc/Datasets/GZ-NPM-Aug
cp -r /content/zestxml/GZXML-Datasets/GZ-NPM xc/Datasets/GZ-NPM
python -c "import apex; print('apex shim ok:', apex.optimizers.FusedAdam)"

In [ ]:
%cd /content/renee
!python -W ignore -u utils/CreateTokenizedFiles.py --data-dir xc/Datasets/GZ-NPM \
  --max-length 32 --tokenizer-type bert-base-uncased --tokenize-label-texts
!python utils/CreateAugData.py --data-dir xc/Datasets/GZ-NPM \
  --tokenization-folder bert-base-uncased-32 --max-len 32

import os
d = 'xc/Datasets/GZ-NPM-Aug/bert-base-uncased-32'
for f in sorted(os.listdir(d)):
    print(f'{f:34s} {os.path.getsize(f"{d}/{f}") // (8 * 32):>7d} rows')
print('expect trn_doc 28350 (25127 points + 3223 label texts), tst_doc 8376, lbl 3223')

In [ ]:
# drop --epochs to 20 for a cheaper first look
!cd /content/renee && python main.py --epochs 50 --batch-size 32 --lr1 0.05 --lr2 1e-5 \
  --warmup 1000 --data-dir xc/Datasets/GZ-NPM-Aug --maxlen 32 \
  --tf sentence-transformers/msmarco-distilbert-base-v4 \
  --dropout 0.85 --pre-tok --wd1 1e-4 --noloss --fp16xfc --expname gznpm-aug

In [ ]:
# Renee's output layout is version dependent: find its matrix, then convert it so the
# same evaluator scores both systems.
import glob, scipy.sparse as sp, torch, sys
sys.path.insert(0, '/content/zestxml')
from zestxml.csr import CSR
from zestxml.io import write_bin_smat, ensure_dir

found = sorted(glob.glob('/content/renee/**/*.npz', recursive=True), key=os.path.getmtime)
print('candidates:', *found, sep='\n  ')
SCORES = found[-1] if found else ''   # or paste a path here

if SCORES:
    m = sp.load_npz(SCORES).tocsr()
    print('renee scores', m.shape, m.nnz)
    ensure_dir('/content/zestxml/Results/renee')
    write_bin_smat(CSR(torch.as_tensor(m.indptr).long(), torch.as_tensor(m.indices).long(),
                       torch.as_tensor(m.data).float(), m.shape),
                   '/content/zestxml/Results/renee/score_mat.bin')

## 6 — the tables

Every row is recomputed from the score matrix on disk, so nothing here can drift from
the artifacts. `--scan` picks up any run not in the script's label list, including
whatever the stages above happened to produce.

In [ ]:
%cd /content/zestxml
!python benchmarks/results_table.py GZ-NPM --scan --md
!python benchmarks/results_table.py GZ-Reuters-90 --scan --md

### Reading the tables

* Sort order is **unseen-label P@1**, which is what these datasets exist to measure.
* An unseen P@1 of **2.49** on npm (2.07 on Reuters) is the evaluator's tie-break floor,
  not a score: those matrices have no non-zero entry in any unseen column. Read it as 0.
  Every per-label classifier — OVA, kNN, centroid — lands there.
* Aggregate P@1 barely separates the ZestXML variants (all within ~0.1 on npm). The
  unseen column is where they differ, by up to 8 points.
* Training is only bit-reproducible at `num_thread=1`; above it, two identical runs differ
  by up to ~0.011 in the metrics. Treat differences of that size as noise.